# Review Meeting Processor
This notebook demonstrates how to process a review meeting video using Azure Content Understanding. It extracts chapters, analyzes valuable knowledge, and summarizes questions, suggestions, and code samples into markdown.

## Import Required Libraries
Import necessary libraries for Azure Content Understanding, file handling, and display.

In [ ]:
# Import Required Libraries
import logging
import json
import os
import sys
import uuid
from pathlib import Path
from dotenv import find_dotenv, load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from IPython.display import display, Markdown, HTML

## Problem Statement
Extract chapters from a review meeting video, analyze valuable knowledge, and summarize questions, suggestions, and code samples for each knowledge item.

In [ ]:
# Setup Azure Content Understanding Client
load_dotenv(find_dotenv())
logging.basicConfig(level=logging.INFO)

AZURE_AI_ENDPOINT = os.getenv("AZURE_AI_ENDPOINT")
AZURE_AI_API_KEY = os.getenv("AZURE_AI_API_KEY")
AZURE_AI_API_VERSION = os.getenv("AZURE_AI_API_VERSION", "2025-05-01-preview")

parent_dir = Path(Path.cwd()).parent
sys.path.append(str(parent_dir))
from python.content_understanding_client import AzureContentUnderstandingClient

credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(credential, "https://cognitiveservices.azure.com/.default")

client = AzureContentUnderstandingClient(
    endpoint=AZURE_AI_ENDPOINT,
    api_version=AZURE_AI_API_VERSION,
    token_provider=token_provider,
    # subscription_key=AZURE_AI_API_KEY,
    x_ms_useragent="azure-ai-content-understanding-python/review_meeting_processor",
 )

In [ ]:
# Specify the review meeting video file
VIDEO_FILE_PATH = Path("../data/AzureSDKReviewMeetingRecording.mp4")

In [ ]:
# Load and print custom analyzer template
analyzer_template_path = "../analyzer_templates/review_meeting_processor.json"
with open(analyzer_template_path, 'r') as f:
    template_content = json.load(f)
    print(json.dumps(template_content, indent=2))

In [ ]:
# Create and run review meeting analyzer
video_analyzer_id = "review_meeting_processor_" + str(uuid.uuid4())
print(f"Creating review meeting analyzer: {video_analyzer_id}")
response = client.begin_create_analyzer(video_analyzer_id, analyzer_template_path=analyzer_template_path)
result = client.poll_result(response)
print("✅ Review meeting analyzer created successfully!")

print(f"Analyzing review meeting video: {VIDEO_FILE_PATH}")
print("⏳ Note: Video analysis may take longer for long meetings...")
response = client.begin_analyze(video_analyzer_id, file_location=VIDEO_FILE_PATH)
result_json = client.poll_result(response, timeout_seconds=360)
print("Review Meeting Content Understanding result: ")
print(json.dumps(result_json, indent=2))

In [ ]:
# Extract and display chapters, knowledge, questions, suggestions, and code samples
def display_review_meeting_results(result_json):
    chapters = result_json.get('chapters', [])
    for idx, chapter in enumerate(chapters):
        topic = chapter.get('topic', 'Unknown')
        begin = chapter.get('beginTimestamp', '-')
        end = chapter.get('endTimestamp', '-')
        is_valuable = chapter.get('isValuable', False)
        display(Markdown(f"### Chapter {idx+1}: {topic} ({begin} - {end})"))
        display(Markdown(f"- Valuable: {'Yes' if is_valuable else 'No'}"))
        knowledge_items = chapter.get('knowledgeExtracted', [])
        for kidx, knowledge in enumerate(knowledge_items):
            desc = knowledge.get('description', '')
            display(Markdown(f"#### Knowledge {kidx+1}: {desc}"))
            questions = knowledge.get('questions', [])
            if questions:
                display(Markdown("**Questions:**"))
                for q in questions:
                    display(Markdown(f"- {q}"))
            suggestions = knowledge.get('suggestions', [])
            if suggestions:
                display(Markdown("**Suggestions:**"))
                for s in suggestions:
                    display(Markdown(f"- {s}"))
            code_samples = knowledge.get('codeSamples', [])
            if code_samples:
                display(Markdown("**Code Samples:**"))
                for c in code_samples:
                    display(Markdown(f"```python\n{c}\n```"))

display_review_meeting_results(result_json)

## Summary Markdown Output
The extracted questions, suggestions, and code samples are summarized above for each chapter and knowledge item. You can copy the markdown output for documentation or further review.